# RAG Step 2: Store Chunks in ChromaDB

Create embeddings for the Step 1 chunks and save them in a persistent ChromaDB collection. Persistent storage lets later notebooks reuse the indexed data. Run Step 1 first in the same Jupyter kernel, or execute the setup cell below.


In [ ]:
# Install once if needed: %pip install chromadb sentence-transformers
import chromadb
from chromadb.utils import embedding_functions

# Step 1 variables should already exist. Recreate a small sample if starting fresh.
if "chunks" not in globals():
    chunks = [
        {"id": "rag_intro_chunk_1", "text": "RAG combines document retrieval with language generation.", "metadata": {"source": "sample/rag_intro.txt", "document_id": "rag_intro", "chunk": 1}},
        {"id": "vector_db_chunk_1", "text": "ChromaDB stores embeddings and supports semantic search.", "metadata": {"source": "sample/vector_db.txt", "document_id": "vector_db", "chunk": 1}},
    ]

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
 )
client = chromadb.PersistentClient(path="./chroma_data")
collection = client.get_or_create_collection(
    name="rag_demo_collection",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
 )

# Upsert makes repeated notebook runs safe for the same chunk IDs.
if chunks:
    collection.upsert(
        ids=[item["id"] for item in chunks],
        documents=[item["text"] for item in chunks],
        metadatas=[item["metadata"] for item in chunks],
    )

print("Collection:", collection.name)
print("Stored record count:", collection.count())
print("Sample IDs:", collection.peek(limit=3)["ids"])


## What happened?

Chroma uses the configured embedding function to turn documents into vectors and persists the vectors with their IDs and metadata. Use the same embedding model when querying. Re-index or remove obsolete chunks if a source document changes. Continue with Step 3 to retrieve passages for a question.